In [ ]:
# 🎨 Análisis de Paleta de Colores usando K-Means
#
# Este notebook implementa un análisis de colores dominantes en una imagen utilizando el algoritmo K-Means.
# Se utiliza el **método de silueta** para determinar el número óptimo de clusters (K).


# ## 1. Importación de Librerías


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from PIL import Image
import warnings

warnings.filterwarnings('ignore')
np.random.seed(42)

print("✅ Librerías cargadas correctamente")


# ## 2. Carga de la Imagen Original


In [ ]:
def cargar_imagen(ruta: str) -> np.ndarray:
    """
    Carga una imagen y la convierte a array RGB.

    Parameters:
    -----------
    ruta : str
        Ruta del archivo de imagen

    Returns:
    --------
    np.ndarray : Array de la imagen (height, width, 3) en uint8
    """
    img = Image.open(ruta)

    if img.mode != 'RGB':
        img = img.convert('RGB')

    return np.array(img, dtype=np.uint8)


In [ ]:
RUTA_IMAGEN = "flores-verano-jardin.jpg"

img_original = cargar_imagen(RUTA_IMAGEN)

print(f"📷 Imagen cargada: {RUTA_IMAGEN}")
print(f"   - Shape: {img_original.shape}")
print(f"   - Total de píxeles: {img_original.shape[0] * img_original.shape[1]:,}")
print(f"   - Tipo de datos: {img_original.dtype}")

# Visualizar la imagen
plt.figure(figsize=(10, 8))
plt.imshow(img_original)
plt.title('Imagen Original', fontsize=14, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()


# ## 3. Preparación de Píxeles


In [ ]:
def preparar_pixeles(img_array: np.ndarray) -> np.ndarray:
    """
    Convierte la imagen a un array de píxeles para clustering.

    Parameters:
    -----------
    img_array : np.ndarray
        Array de la imagen (height, width, 3)

    Returns:
    --------
    np.ndarray : Array de píxeles (n_pixeles, 3)
    """
    return img_array.reshape(-1, 3).astype(np.float64)


In [ ]:
pixels = preparar_pixeles(img_original)
print(f"📊 Píxeles preparados: {len(pixels):,}")


# ## 4. Análisis del Método de Silueta


In [ ]:
def analizar_silhouette(pixels: np.ndarray, k_min: int = 2, k_max: int = 12,
                        sample_size: int = 10000) -> dict:
    """
    Analiza el coeficiente de silueta para diferentes valores de K.
    Usa una muestra de píxeles para calcular silueta (eficiencia).

    Parameters:
    -----------
    pixels : np.ndarray
        Array de píxeles (n_pixeles, 3)
    k_min : int
        Valor mínimo de K
    k_max : int
        Valor máximo de K
    sample_size : int
        Tamaño de muestra para calcular silueta

    Returns:
    --------
    dict : Resultados del análisis
    """
    resultados = {
        'k_values': [],
        'silhouette_scores': [],
        'inertias': [],
        'modelos': []
    }

    # Muestra para silueta (si hay más píxeles que sample_size)
    if len(pixels) > sample_size:
        indices = np.random.choice(len(pixels), sample_size, replace=False)
        pixels_sample = pixels[indices]
    else:
        pixels_sample = pixels

    print("🔍 Analizando método de silueta...")
    print("-" * 50)

    for k in range(k_min, k_max + 1):
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300)
        labels = kmeans.fit_predict(pixels_sample)
        score = silhouette_score(pixels_sample, labels)

        resultados['k_values'].append(k)
        resultados['silhouette_scores'].append(score)
        resultados['inertias'].append(kmeans.inertia_)
        resultados['modelos'].append(kmeans)

        print(f"   K = {k:2d} → Silueta: {score:.4f}")

    mejor_idx = np.argmax(resultados['silhouette_scores'])
    resultados['mejor_k'] = resultados['k_values'][mejor_idx]
    resultados['mejor_score'] = resultados['silhouette_scores'][mejor_idx]

    print("-" * 50)
    print(f"✅ Mejor K: {resultados['mejor_k']} (Silueta: {resultados['mejor_score']:.4f})")

    return resultados


In [ ]:
K_MIN = 2
K_MAX = 12

resultados = analizar_silhouette(pixels, k_min=K_MIN, k_max=K_MAX)


# ## 5. Visualización del Análisis de Silueta


In [ ]:
def visualizar_analisis(resultados: dict) -> None:
    """Visualiza los resultados del análisis de silueta."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    k_values = resultados['k_values']
    scores = resultados['silhouette_scores']
    inertias = resultados['inertias']
    mejor_k = resultados['mejor_k']
    mejor_idx = k_values.index(mejor_k)

    # Método de Silueta
    bars = axes[0].bar(k_values, scores, color='steelblue', edgecolor='navy', alpha=0.7)
    bars[mejor_idx].set_color('forestgreen')
    bars[mejor_idx].set_alpha(1.0)
    axes[0].axhline(y=resultados['mejor_score'], color='red', linestyle='--', alpha=0.5)
    axes[0].set_xlabel('K', fontsize=11)
    axes[0].set_ylabel('Coeficiente de Silueta', fontsize=11)
    axes[0].set_title('📊 Método de Silueta', fontsize=13, fontweight='bold')
    axes[0].set_xticks(k_values)

    # Método del Codo
    axes[1].plot(k_values, inertias, 'o-', linewidth=2, markersize=8, color='coral')
    axes[1].axvline(x=mejor_k, color='forestgreen', linestyle='--', alpha=0.7)
    axes[1].set_xlabel('K', fontsize=11)
    axes[1].set_ylabel('Inercia', fontsize=11)
    axes[1].set_title('📈 Método del Codo', fontsize=13, fontweight='bold')
    axes[1].set_xticks(k_values)

    plt.tight_layout()
    plt.show()


In [ ]:
visualizar_analisis(resultados)


# ## 6. Extracción de Paleta de Colores


In [ ]:
def extraer_paleta(pixels: np.ndarray, k: int) -> tuple:
    """
    Extrae la paleta de colores usando K-Means.

    Returns:
    --------
    tuple : (colores, porcentajes, labels, kmeans)
    """
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300)
    labels = kmeans.fit_predict(pixels)

    colores = np.clip(kmeans.cluster_centers_, 0, 255).astype(np.uint8)

    unique, counts = np.unique(labels, return_counts=True)
    porcentajes = (counts / len(labels)) * 100

    # Ordenar por porcentaje
    orden = np.argsort(porcentajes)[::-1]

    return colores[orden], porcentajes[orden], labels, kmeans

def rgb_to_hex(rgb: np.ndarray) -> str:
    return '#{:02X}{:02X}{:02X}'.format(rgb[0], rgb[1], rgb[2])


In [ ]:
mejor_k = resultados['mejor_k']
colores, porcentajes, labels, kmeans = extraer_paleta(pixels, mejor_k)

print(f"🎨 Paleta de {mejor_k} colores:")
print("-" * 30)
for i, (color, pct) in enumerate(zip(colores, porcentajes)):
    print(f"   {rgb_to_hex(color)} → {pct:.1f}%")


# ## 7. Visualización Final


In [ ]:
def visualizar_resultado(img_original: np.ndarray, colores: np.ndarray,
                         porcentajes: np.ndarray, kmeans) -> None:
    """Visualiza la imagen original, segmentada y la paleta."""

    fig = plt.figure(figsize=(16, 10))
    gs = fig.add_gridspec(2, 3, height_ratios=[2, 1], hspace=0.3, wspace=0.2)

    # Imagen original
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.imshow(img_original)
    ax1.set_title('📷 Imagen Original', fontsize=12, fontweight='bold')
    ax1.axis('off')

    # Imagen segmentada
    ax2 = fig.add_subplot(gs[0, 1])
    pixels_flat = img_original.reshape(-1, 3).astype(np.float64)
    labels_img = kmeans.predict(pixels_flat)
    colores_centroides = kmeans.cluster_centers_.astype(np.uint8)
    img_segmentada = colores_centroides[labels_img].reshape(img_original.shape)
    ax2.imshow(img_segmentada)
    ax2.set_title(f'🎯 Imagen Segmentada (K={len(colores)})', fontsize=12, fontweight='bold')
    ax2.axis('off')

    # Paleta de colores
    ax3 = fig.add_subplot(gs[0, 2])
    ax3.axis('off')
    ax3.set_title('🎨 Paleta de Colores', fontsize=12, fontweight='bold')

    n_colores = len(colores)
    for i, (color, pct) in enumerate(zip(colores, porcentajes)):
        y = 1 - (i + 1) * (0.9 / n_colores)
        rect = plt.Rectangle((0.1, y), 0.8, 0.8 / n_colores,
                             facecolor=color/255, edgecolor='black', linewidth=2,
                             transform=ax3.transAxes)
        ax3.add_patch(rect)

        hex_color = rgb_to_hex(color)
        luminancia = 0.299 * color[0] + 0.587 * color[1] + 0.114 * color[2]
        text_color = 'white' if luminancia < 128 else 'black'
        ax3.text(0.5, y + 0.4 / n_colores, f'{hex_color} ({pct:.1f}%)',
                transform=ax3.transAxes, ha='center', va='center',
                fontsize=10, fontweight='bold', color=text_color)

    # Barra de proporción
    ax4 = fig.add_subplot(gs[1, :])
    left = 0
    for color, pct in zip(colores, porcentajes):
        width = pct / 100
        ax4.barh(0, width, left=left, color=color/255, edgecolor='white', height=0.6)
        if pct > 5:
            luminancia = 0.299 * color[0] + 0.587 * color[1] + 0.114 * color[2]
            text_color = 'white' if luminancia < 128 else 'black'
            ax4.text(left + width/2, 0, f'{pct:.1f}%', ha='center', va='center',
                    fontsize=10, fontweight='bold', color=text_color)
        left += width

    ax4.set_xlim(0, 1)
    ax4.set_ylim(-0.5, 0.5)
    ax4.axis('off')
    ax4.set_title('📊 Proporción de Colores', fontsize=12, fontweight='bold', pad=20)

    plt.suptitle(f'Análisis de Paleta - K Óptimo: {len(colores)}',
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()


In [ ]:
visualizar_resultado(img_original, colores, porcentajes, kmeans)


# ## 📋 Resumen


In [ ]:
print("=" * 50)
print("📋 RESUMEN DEL ANÁLISIS")
print("=" * 50)
print(f"   🖼️  Imagen: {RUTA_IMAGEN}")
print(f"   🔢  Rango K evaluado: {K_MIN} - {K_MAX}")
print(f"   ✅  K óptimo: {resultados['mejor_k']}")
print(f"   📊  Silueta: {resultados['mejor_score']:.4f}")
print(f"   🎨  Colores extraídos: {len(colores)}")
print("=" * 50)
